# Point-in-time feature engineering
The production path is `python -m src.data.pipeline`. This notebook reviews its output and verifies the cutoff, without replacing the scripted pipeline.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
import json
metadata = json.loads((root / 'data/processed/feature_metadata.json').read_text())
display(metadata)
data = pd.read_parquet(root / 'data/processed/customers.parquet')
display(data.head())
assert data.customer_id.is_unique
display(data.groupby('split').agg(customers=('customer_id','size'), churn_rate=('churn','mean')))

{'cutoff_exclusive': '2011-09-01 00:00:00',
 'horizon_days': 90,
 'customers': 5224,
 'split_counts': {'train': 3656, 'validation': 784, 'test': 784},
 'churn_rate': 0.5719754977029097,
 'category_definition': 'Ordered description keyword proxies; no official categories exist',
 'discount_definition': "Share of lines below the same customer's earlier maximum SKU price; not a causal promotion response"}

,customer_id,monetary,recency,tenure,frequency,average_spend,basket_size,gap_variance,frequency_trend,seasonal_concentration,...,affinity_bags,affinity_stationery,affinity_toys,affinity_other,engagement_score,churn,next_purchase_days_capped,clv_90d,next_category_proxy,split
0,12346,77352.96,225.582639,547.452778,3,25784.320,24746.333333,1840.737723,0.000000,0.997811,...,0.000000,0.000000,0.000000,0.998160,42.904023,1,90.000000,0.00,no_purchase,validation
1,12347,3402.39,29.633333,304.402778,6,567.065,349.833333,133.231612,0.085714,0.209203,...,0.134846,0.108630,0.002963,0.327784,57.474998,0,60.517361,1294.32,kitchen,train
2,12348,1388.40,148.550694,338.375694,4,347.100,622.000000,296.358154,-0.085714,0.470182,...,0.000000,0.315471,0.012244,0.362432,29.580232,0,24.550694,270.00,other,test
3,12349,2221.14,307.650694,489.444444,2,1110.570,495.500000,0.000000,0.000000,0.518932,...,0.010805,0.037278,0.007969,0.512142,24.187592,0,81.410417,1457.55,other,train
4,12350,294.40,210.332639,210.332639,1,294.400,196.000000,0.000000,0.000000,1.000000,...,0.000000,0.000000,0.000000,0.930707,10.527858,1,90.000000,0.00,no_purchase,train


,customers,churn_rate
split,,
test,784,0.571429
train,3656,0.571937
validation,784,0.572704


In [2]:
events = pd.read_parquet(root / 'data/processed/purchase_sequences.parquet')
assert events.invoice_date.max() < pd.Timestamp(metadata['cutoff_exclusive'])
display(events.head())

,customer_id,invoice,invoice_date,amount,units,category_proxy,gap_days
0,12346,499763,2010-03-02 13:08:00,27.05,5,other,0.000000
1,12346,513774,2010-06-28 13:53:00,142.31,19,home,118.031250
2,12346,541431,2011-01-18 10:01:00,77183.60,74215,other,203.838889
3,12347,529924,2010-10-31 14:20:00,611.53,509,kitchen,0.000000
4,12347,537626,2010-12-07 14:57:00,711.79,319,other,37.025694


## Feature definitions
Recency, distinct orders, total and average product spend, basket units, six-month order trend, gap variance, seasonal concentration, return rate, description-based affinity and price-based discount proxy use pre-cutoff records only. The engagement score combines fixed recency, order and spend scales. Historical CLV is observed net product revenue; predicted CLV means the next 90 days only. It is not a full lifetime forecast. Category and discount proxies do not represent official categories or known promotions.